# AegisFlow: Modelagem Matemática e Complexidade Algorítmica de um Firewall Dinâmico

Este documento consolida o rigor matemático por trás das operações algorítmicas do sistema AegisFlow. O código é tratado como a execução sistêmica de provas matemáticas.


## 1. Teoria dos Conjuntos: Fusão de Blocklists e Unicidade de Acesso

### 1.1. Contextualização do Cenário (Cibersegurança)
No AegisFlow, recebemos listas de IPs maliciosos (*blocklists*) de múltiplas agências de inteligência. A união bruta destas listas, invariavelmente resulta em redundância (IPs listados por mais de uma agência). Em um sistema de tempo real, dados redundantes representam desperdício de alocação contígua de memória e dilatação do tempo de busca ($O(n)$) no pior caso. A exigência arquitetural é a consolidação de um conjunto estrito com cardinalidade real.

### 1.2. Modelagem matemática
Sejam dois conjuntos de IPs maliciosos, onde $A$ representa a lista da agência 1 e $B$ a lista da agência 2.

A operação fundamental para a consolidação das listas é a **União de Conjuntos**, definida rigorsamente como:

$$ A \cup B = \{x \mid x \in A \lor x \in B\} $$

Lê-se:
"O conjunto de $A$ união $B$ é formado por todo elemento $x$, tal que ($|$) $x$ pertence ($\in$) ao conjunto de $A$ ou ($\lor$) $x$ pertence ao conjunto $B$.

No entanto, a definição matemática de um conjunto exige que todos os seus elementos sejam distintos. Se houver elementos que pertençam tanto a $A$ quanto a $B$, eles formam a **Intersecção**:

$$ A \cap B = \{x \mid x \in A \land x \in B\} $$

Lê-se:
"O conjunto $A$ intersecção $B$ é formado por todo elemento $x$, tal que $x$ pertence a $A$ e ($\land$) $x$ pertence a $B$.

### 1.3. O Desafio Algorítmico (Justificativa de implementação)
Linguagens de alto nível mascaram o custo dessas operações através de *Hash Tables* (como a função `set()` em Python). Para forjar a fundação lógica, a implementação algorítmica exigirá a tradução literal da notação matemática usando apenas arranjos lineares (listas) e estruturas de repetição brutas. Isso nos forçará a lidar com a complexidade de tempo de verificação de pertinência antes de cada inserção. Essa realidade traduziria-se da seguinte forma em Python:

In [3]:
agencia_1: list[str] = [
    "192.168.1.10",
    "192.168.1.20",
    "10.0.0.5",
    "172.16.0.10",
    "8.8.8.8"
    ]
agencia_2: list[str] = [
    "10.0.0.5",
    "192.168.1.20",
    "10.0.0.15",
    "172.16.0.20",
    "1.1.1.1",
]

copia_base = agencia_1.copy()

for ip_suspeito in agencia_2:
    if ip_suspeito not in copia_base:
        copia_base.append(ip_suspeito)

print(copia_base)

['192.168.1.10', '192.168.1.20', '10.0.0.5', '172.16.0.10', '8.8.8.8', '10.0.0.15', '172.16.0.20', '1.1.1.1']


### 1.4. Análise Assintótica do Motor de Fusão (Abordagem Ingênua)

#### 1.4.1. O Custo Operacional Oculto
A implementação inicial da fusão de conjuntos utilizou a expressão `not in` do Python sobre uma estrutura linear (Lista). Embora sintaticamente elegante, essa abordagem esconde um custo computacional severo.

Seja $N$ o tamanho do conjunto base ($A$) e $M$ o tamanho do conjunto de novas regras ($B$). O motor de iteração principal executa $M$ ciclos. Em cada ciclo, a verificação de pertinência (`not in`) exige uma **Busca Sequencial**.

#### 1.4.2. A Prova Matemática do Gargalo
No pior caso (intersecção nula, onde cada IP deve ser inspecionado até o final da lista sem sucesso e depois anexado), o número de comparações para a inserção do elemento $i$ de $B$ cresce de forma progressiva. O custo total de operações $C$ é definido por:

$$ C = \sum_{i=1}^{M} {N + i - 1} $$

O limite assintótico que descreve esse crescimento na notação Big-O é:

$$ O(M \times N) $$

Assumindo um cenário onde o volume de dados em ambas as listas escala de forma proporcional ($M \approx N$), o algoritmo atinge uma complexidade de tempo quadrática:

$$ O(n^2) $$

**Conclusão:** Esta implementação é inaceitável para sistemas de tempo real. Ela foi documentada unicamente para estabelecer a base argumentativa de refatoração futura, evidenciando o gargalo de processamento gerado pelo desconhecimento da estrutura de dados subjacente.

## 2. Teoria das Filas e o Gargalo de Buffering (Modelo FIFO)

### 2.1. Modelagem Estocástica do Tráfego (Lei de Little)
No AegisFlow, as requisições de rede (pacotes) não são ispecionados instantaneamente assim que atingem a interface física; elas chegam de forma estocástica e precisam ser acomodadas em um *bugger* de memória. O comportamento desse estrangulamento é modelado matematicamente pela **Teoria das Filas**.

A equação fundamental que rege a estabilidade do nosso *buffer* é a **Lei de Little**:

$$ L = \lambda W $$

Onde:
* $L$ representa o número médio de requisições retidas no sistema (o tamanho de nossa fila).
* $\lambda$ (Lambda) representa a taxa média de chegada de pacotes (requisições por segundo).
* $W$ representa o tempo médio total que um pacote gasta no sistema (o tempo gasto esperando na fila somado ao tempo gasto sendo processado pelas nossas regras de bloqueio).

**Interpretação para o AegisFlow:** O nosso motor de fusão inicial possui complexidade $O(n^2)$. Isso aumenta drasticamente o tempo de processamento das regras, o que eleva $W$. Pela Lei de Little, se $\lambda$ permaneer constante e $W$ aumentar, $L$ (o tamanho do *buffer*) crescerá indefinidamente até causar um *buffer overflow* ou esgotamento total de memória (OOM - *Out of Memory*).

### 2.2. Estrutura FIFO (First in, First out) e o Custo de Mutação Indexada
A política de atendimento de requisições de um firewall padrão é **FIFO** (o primeiro a entrar é o primeiro a ser processado).

Se abstrairmos essa fila usando uma Lista linear simples, teremos a seguinte disfunção arquitetural:
* **Enfileirar (Enqueue):** Adicionar um elemento ao final do bloco de memória alocado possui tempo constante, $O(1)$ amortizado.
* **Desenfileirar (Dequeue):** O consumo do pacote na ponta inicial da fila (índice 0) cria um vácuo na memória. O interpretador é forçado a deslocar todos os $n-1$ pacotes restantes uma posição para a esquerda. O tempo de consumo se torna linear: $O(n)$.

**Conclusão algorítmica:** O uso de listas nativas para operações de fila de alta frequência adiciona uma latência $O(n)$ a cada pacote consumido, provando ser mais um antipadrão que será documentado antes de aplicarmos a estrutura de dados correta. Note abaixo a representação em Python deste antipadrão:

In [2]:
agencia_1: list[str] = [
    "192.168.1.10",
    "192.168.1.20",
    "10.0.0.5",
    "172.16.0.10",
    "8.8.8.8"
    ]
agencia_2: list[str] = [
    "10.0.0.5",
    "192.168.1.20",
    "10.0.0.15",
    "172.16.0.20",
    "1.1.1.1",
]

copia_base = agencia_1.copy()

for ip_suspeito in agencia_2:
    if ip_suspeito not in copia_base:
        copia_base.append(ip_suspeito)

buffer_rede: list[str] = [
                        "192.168.1.10",
                        "192.168.1.20",
                        "10.0.0.5",
                        "172.16.0.10",
                        "203.0.113.42",
                        "192.168.1.30",
                        ]

while buffer_rede:
    ip_analisado = buffer_rede.pop(0)
    if ip_analisado in copia_base:
        print(f"Pacote bloqueado: {ip_analisado}")
    else:
        print(f"Pacote liberado: {ip_analisado}")

Pacote bloqueado: 192.168.1.10
Pacote bloqueado: 192.168.1.20
Pacote bloqueado: 10.0.0.5
Pacote bloqueado: 172.16.0.10
Pacote liberado: 203.0.113.42
Pacote liberado: 192.168.1.30


## 3. Teoria da Computação: Modelagem Algébrica do Autômato com Pilha (LIFO)

### 3.1. A Insuficiência das Expressões Regulares
A validação de escopos perfeitamente balanceados, como parênteses `()`, é um problema que pertence à classe das **Linguagens Livres de Contexto**. Para resolver aninhamnentos arbitrários, a Teoria da Computação exige o uso de um **Autômato com Pilha** (*Pushdown AutomaTon* - PDA).

### 3.2. Definição formal do Autômato (A matemática pura)
Um autômato com Pilha é formalmente definido pelo 7-upla:

$$ M = (Q, \Sigma, \Gamma, \delta, q_0, Z_0, F) $$

Onde, para o AegisFlow validar a cadeia de *payload*:
* $Q = \{q_0, q_f\}$: Conjunto finito de estados (início e fim).
* $\Sigma = \{(, )\}$: O alfabeto de entrada (os caracteres do *payload*).
* $\Gamma = \{(,Z_0\}$: O alfabeto da pilha, onde $Z_0$ é o símbolo inicial que marca a pilha vazia.
* $q_0$: Estado inicial.
* $Z_0$: Símbolo de base da pilha.
* $F = \{q_f\}$: Conjunto de estados de aceitação.
* $\delta$: A função de transição, $\delta: Q \times (\Sigma \cup \{\epsilon\}) \times \Gamma \rightarrow \mathcal{P}(Q \times \Gamma^*)$, definida pelas seguintes regras estritas:

**Regra de empilhamento (Push):**
Ao ler o caractere de abertura `(`, mapeamos o estado para adicionar um elemento ao topo da memória:
$$ \delta(q_0, \text{'('}, Z_0) = \{(q_0, \text{'('} Z_0)\} $$
$$ \delta(q_0, \text{'('}, \text{'('}) = \{(q_0, \text{'('} \text{'('})\} $$

**Regra de Desempilhamento (Pop):**
Ao ler o caractere de fechamento `)`, mapeamos o estado para consumir o elemento do topo (representado por $\epsilon$, denotando a remoção):
$$ \delta(q_0, \text{')'}, \text{')'}) = \{(q_0, \epsilon)\} $$

**Regra de Aceitação e Underflow:**
O autômato aceita a cadeia se terminar a leitura da string e restar apenas $Z_0$ na pilha:
$$ \delta(q_0, \epsilon, Z_0) = \{(q_f, Z_0)\} $$
Se a função de transição tentar executar um pop sobre $Z_0$ antes da cadeia acabar (excesso de fechamento), a função $\delta$ é indefinida e o autômato colapsa (estado de erro matemático, traduzindo no Python como `IndexError`).

In [1]:
pilha_escopo: list[str] = []
payload_suspeito: str = "( ( DROP TABLE users ) ( )"
anomalia_detectada = False

for char in payload_suspeito:
    if char == "(":
        pilha_escopo.append(char)
    elif char == ")":
        if pilha_escopo:
            pilha_escopo.pop()
        else:
            anomalia_detectada = True
            break

if not pilha_escopo and not anomalia_detectada:
    print("Payload válido.")

else:
    print("Anomalia de escopo.")

Anomalia de escopo.


## 4. Estatística Descritiva: O Supremo de um Conjunto Linear (Tempo O(n))

### 4.1. O Problema da Extração sem Ordenação
No AegisFlow, o módulo da inteligência quantifica a agressividade dos pacotes em um vetor estritamente linear, denotado por $V$.

$$ V = (v_1, v_2, \dots, v_n)$$

A linguagem Python oferece abstrações de alto nível (como a função `max()` ou o método `sort()`) para encontrar o maior valor. Contudo, a fundação algorítmica exige a prova de como esse supremo é extraído mecanicamente da memória. A ordenação prévia do vetor custaria, no melhor dos casos para listas nativas, $O(n \log n)$ com o *Timsort*. Para encontrar apenas o limite superior, a matemática exige uma única passagem iterativa com complexidade de tempo $O(n)$.

### 4.2. A Relação de Recorrência (Mutação de Estado)
Para localizar o valor máximo $M$ em $V$ sem ordenar os elementos, o processador deve reter um estado temporário em memória que represente "o maior valor encontrado até o momento" ($m_k$), atualizando-o condicionalmente a cada novo ciclo da máquina.

A Relação de Recorrência para o estado $m_k$ no passo $k$ é definida matematicamente por:

**Passo Base (Inicialização):**
O estado inicial assume o valor do primeiro elemento do conjunto.
$$ m_1 = v_1 $$

**Passo Indutivo (Transição de Estado):**
Para todo $k > 1$, o estado temporário é confrontado com o elemento atual do vetor.

$$m_k = \begin{cases}
m_{k-1}, & \text{se } m_{k-1} \ge v_k \\
v_k, & \text{se } m_{k-1} < v_k
\end{cases} $$

### 4.3. O Custo Operacional Absoluto
O algoritmo realizará exatamente $n - 1$ comparações no Passo Indutivo. A complexidade de espaço é estritamente $O(1)$, pois apenas o ponteiro do estado $m_k$ é alocado, e a complexidade de tempo é linear $O(n)$.

Em Python, traduz-se na seguinte estrutura:

In [2]:
niveis_ameaca: list[int] = [45, 89, 12, 99, 34]

if niveis_ameaca:
    maior_ameaca = niveis_ameaca[0]

    for ameaca in niveis_ameaca[1:]:
        if ameaca > maior_ameaca:
            maior_ameaca = ameaca

    print(f"Maior nível de ameaça detectado: {maior_ameaca}")

print("Tráfego limpo. Nenhuma ameaça registrada no ciclo.")

Maior nível de ameaça detectado: 99
Tráfego limpo. Nenhuma ameaça registrada no ciclo.


## 5. Ordenação Quadrática: A Mecânica do Bubble Sort e o Custo $O(n^2)$

### 5.1. O Teorema da Ordenação Adjacente
Dado um vetor de logs $V$ não ordenado, de cardinalidade $n$:
$$V = (v_1, v_2, \dots, v_n)$$

O objetivo da ordenação é permutar os elementos de $V$ de modo a atingir a monotonicidade crescente, onde cada elemento é menor ou igual ao seu sucessor:
$$v_1 \le v_2 \le \dots \le v_n$$

O *Bubble Sort* atinge este estado sem alocar nova memória (opera *in-place*, complexidade de espaço $O(1)$) através de varreduras sucessivas, comparando estritamente pares adjacentes $(v_k, v_{k+1})$. Se a condição $v_k > v_{k+1}$ for verdadeira, ocorre a permutação algébrica (*Swap*).

### 5.2. A Matemática do Gargalo: A Progressão Aritmética
A ineficiência deste algoritmo não reside na operação de troca, mas na exaustão das comparações. Em uma única passagem linear, apenas o maior elemento do subconjunto atual "flutua" garantidamente para a sua posição final à direita. Portanto, para ordenar $n$ elementos, o processador é obrigado a realizar $n-1$ varreduras completas.

Na primeira varredura, o algoritmo executa $n-1$ comparações. Na segunda, a última posição já está ordenada, restando $n-2$ comparações. Isso decresce linearmente até restar apenas 1 comparação. 

O custo total de operações $C$ é a soma dessa progressão aritmética:
$$C = (n-1) + (n-2) + \dots + 1 = \sum_{i=1}^{n-1} i$$

Aplicando a soma da PA, obtemos a função polinomial exata do processamento:
$$C = \frac{n(n-1)}{2} = \frac{n^2 - n}{2}$$

Na Análise Assintótica, descartamos os divisores constantes e os termos de menor grau. A matemática prova que o limite superior do tempo de execução degenera de forma estritamente quadrática:
$$O(n^2)$$

Em Python, vemos sua aplicação:

In [3]:
niveis_ameaca: list[int] = [45, 89, 12, 99, 34]
n = len(niveis_ameaca)

for i in range(n - 1):
    for j in range((n - 1) - i):
        if niveis_ameaca[j] > niveis_ameaca[j + 1]:
            niveis_ameaca[j], niveis_ameaca[j + 1] = niveis_ameaca[j + 1], niveis_ameaca[j]

print(niveis_ameaca)

[12, 34, 45, 89, 99]


## 6. Estatística Descritiva: A Matemática da Mediana (Tendência Central)

### 6.1. A Dependência da Monoticidade
Diferente da média aritmética, que sofre distorções massivas por valores extremos (anomalias de tráfego), a Mediana ($Me$) exige que o conjunto $V$ de cardinalidade $n$ esteja estritamente ordenado:
$$ v_1 \le v_2 \dots \le v_n$$

A Mediana é o valor que divide o conjunto exatamente ao meio. A sua localização espacial na memória depende da paridade geométrica de $n$.

### 6.2. O Axioma da Paridade
A máquina deve ramificar a sua lógica em dois estados matemáticos distintos, baseado na cardinalidade de $n$:

**Caso 1: $n$ é Ímpar**
Se o número de elementos for ímpar, existe um único elemento central absoluto. A coordenada matemática exata (assumindo índice base 0) é a divisão inteira de $n$ por 2.
$$ Me = v_{\lfloor n/2 \rfloor} $$

**Caso 2: $n$ é Par**
Se o número de elementos for par, o centro do conjunto cai no vazio entre dois índices. A matemática exige a extração da média aritmética estrita entre os dois valores centrais.
$$ Me = \frac{v_{(n/2) - 1} + v_{n/2}}{2} $$

Em Python, temos:

In [4]:
niveis_ameaca: list[int] = [45, 89, 12, 99, 34]
n = len(niveis_ameaca)

# A mediana requer os elementos pré-ordenados
for i in range(n - 1):
    for j in range((n - 1) - i):
        if niveis_ameaca[j] > niveis_ameaca[j + 1]:
            niveis_ameaca[j], niveis_ameaca[j + 1] = niveis_ameaca[j + 1], niveis_ameaca[j]

if n % 2 == 0:
    mediana = (niveis_ameaca[n // 2 - 1] + niveis_ameaca[n // 2]) / 2
else:
    mediana = niveis_ameaca[n // 2]

## 7. Estatística Descritiva: A Matemática da Moda e o Mapeamento de Frequência

### 7.1. O Agrupamento por Monotonicidade
Seja um multiconjunto de logs $V = (v_1, v_2, \dots, v_n)$ previamente ordenado de forma monotônica crescente ($v_1 \le v_2 \le \dots \le v_n$).
Neste estado estrutural, todos os eventos da mesma magnitude estão alocados em posições contíguas (vizinhas) na memória física.

### 7.2. Maximização de Subsequência Contígua
A busca pela Moda ($Mo$) em um vetor ordenado reduz-se ao rastreamento do comprimento da maior subsequência de elementos idênticos.
O processador varre a matriz sequencialmente e compara o elemento atual $v_i$ com o seu predecessor $v_{i-1}$.
- Se $v_i = v_{i-1}$, o agrupamento continua, e a frequência do bloco atual sofre incremento.
- Se $v_i \neq v_{i-1}$, a fronteira do bloco foi rompida, a contagem recomeça.

A Moda é atualizada sempre que o agrupamento atual supera o agrupamento máximo histórico registrado pelo sistema. A complexidade de tempo despenca para $O(n)$ e a complexidade espacial mantém-se absoluta em $O(1)$.

Em Python temos:

In [5]:
niveis_ameaca: list[int] = [45, 89, 12, 99, 34]
n = len(niveis_ameaca)

for i in range(n - 1):
    for j in range((n - 1) - i):
        if niveis_ameaca[j] > niveis_ameaca[j + 1]:
            niveis_ameaca[j], niveis_ameaca[j + 1] = niveis_ameaca[j + 1], niveis_ameaca[j]

moda = niveis_ameaca[0]
frequencia_maxima = 1
valor_ameaca = niveis_ameaca[0]
contador_frequencia = 1

for ameaca in niveis_ameaca[1:]:
    if ameaca == valor_ameaca:
        contador_frequencia += 1
    else:
        if contador_frequencia > frequencia_maxima:
            frequencia_maxima = contador_frequencia
            moda = valor_ameaca

        valor_ameaca = ameaca
        contador_frequencia = 1

if contador_frequencia > frequencia_maxima:
    frequencia_maxima = contador_frequencia
    moda = valor_ameaca

O bloco de código acima apresenta uma limitação: é extraída apenas a primeira moda estatística encontrada em uma sequência linear. Isso acontece por conta do operador relacional "maior que" ($>$) no último condicionante que impede que diversos valores encontrados sob a mesma frequência englobem a Moda Estatística. Ao avançarmos as próximas páginas, desenvolveremos os conceitos de Distribuição Bimodal e Multimodal no escopo matemático e a sua aplicação em código Python, que possibilitarão ampliar o conhecimento tanto matemático puro como a aplicação em código Python.

## 8. Estatística Descritiva: A Média Aritmética e o Centro Gravitacional

### 8.1. A Distribuição da Magnitude
A **Média Aritmética** ($\mu$ para populações ou $\bar{x}$ para amostras) é a medida de tendência central que distribui o "peso" total do sistema igualmente entre todos os seus elementos. Difrentemente da Mediana, que avalia estritamente a *posição* geométrica, a Média é sensível à magnitude de cada evento individual. Um único log de ataque com magnitude extrema (um *outlier*) puxará o centro da gravidade do sistema para si.

### 8.2. A Formulação Algébrica
Seja um multiconjunto de logs $V = \{v_1, v_2, \dots, v_n\}$ de cardinalidade $n$. A média é definida pela razão entre o somatório de todos os valores extraídos e a cardinalidade do conjunto:
$$ \bar{x} = \frac{1}{n} \sum_{i=1}^{n} v_i $$

É incrivelmente simples como em Python podemos traduzir a abordagem matemática:

In [6]:
niveis_ameaca: list[int] = [45, 89, 12, 99, 34]
acumulador = 0

for ameaca in niveis_ameaca:
    acumulador += ameaca

if not niveis_ameaca:
    media = 0
else:
    media = acumulador / len(niveis_ameaca)

## 9. Estatística Descritiva: A Dispersão do Caos

### 9.1. Variância e a Punição de Outliers
A **Variância** ($\sigma^2$) quantifica o quão espalhado está o tráfego em relação ao centro gravitacional (Média). Para evitar que desvios negativos e positivos se anulem, a diferença entre cada log e a média é elevada ao quadrado. Isso não apenas resolve o problema dos sinais, mas também pune matematicamente os ataques extremos, tornando a anomalia exponencialmente mais visível no cálculo.

A formulação algébrica da Variância populacional é:
$$ \sigma^2 = \frac{1}{n} \sum_{i=1}^{n} (v_i - \bar{x})^2 $$

### 9.2. Desvio Padrão e Retorno à Escala
A Variância distorce a escala original da métrica. Para trazer o número de volta à dimensão física real do tráfego e permitir a criação de limites de alerta (thresholds) no nosso sistema de defesa, extraimos a raiz quadrada da Variância. Esse é o **Desvio Padrão** ($\sigma$).

$$ \sigma = \sqrt{\sigma^2} $$

Veja a implementação em Python:

In [7]:
niveis_ameaca: list[int] = [45, 89, 12, 99, 34]
acumulador = 0

for ameaca in niveis_ameaca:
    acumulador += ameaca

if not niveis_ameaca:
    media = 0
else:
    media = acumulador / len(niveis_ameaca)

sigma_acumulador = 0

for ameaca in niveis_ameaca:
    sigma_acumulador += (ameaca - media) ** 2

if not niveis_ameaca:
    variancia = 0
    desvio_padrao = 0
else:
    variancia = sigma_acumulador / len(niveis_ameaca)
    desvio_padrao = variancia ** 0.5